# Programming Assignment 6: Sentiment Analysis
**TripAdvisor hotel reviews - lexicon-based and machine learning-based, step by step**

## 📋 Assignment Information

**Course:** Unstructured Data Analysis (2026-2)
**Assignment:** Programming Assignment 6 - Sentiment Analysis
**Released:** November 23, 2026 (Mon), 09:00
**Deadline:** November 29, 2026 (Sun), 23:59 (late submissions receive 0 points)
**Total:** 100 points

## 🎯 Objectives

Practice the code of the two Week 12 notebooks, one small block at a time:

- **Lexicon-based approach** (lecture 10-2 and lecture 11-1 / 11-2, chapter 3): labels from star ratings, counting opinion
  words, evaluation, growing a lexicon from WordNet, and the corpus-based t-test of a word
- **Machine learning-based approach** (lecture 11-1 / 11-2, chapter 4): logistic regression with bigrams, the Elastic Net
  lexicon, and label spreading on the lecture's word graph

Five **short-answer questions** sit between the TODO items, each with two 2-point items. They ask *why* a result looks the
way it does or ask for one value computed by hand, and they are the kind of question you will meet in the final exam.

Everything is inside the lectures and the two notebooks. The data is a **different** sample of the same hotel reviews
(3,000 training and 1,000 test reviews), so your numbers differ from the class notebooks.

## 📝 Instructions

1. `File ▸ Save a copy in Drive`, then rename the notebook to **`Assignment_6_YourName_StudentID.ipynb`** (e.g. `Assignment_6_HongGilDong_2026123456.ipynb`).
2. Fill in every `________` blank. Each blank is short: a name, a number, an operator or a short expression.
   **Do not** change the code outside the blanks and **do not** rename variables: the grader re-runs your notebook.
3. Every TODO ends with a **Check** line: your output should match it (a difference only in the last digit is not penalized).
4. Answer every **Short answer** in the markdown cell below it (double-click the cell to edit). Keep every short answer
   to one or two sentences and write it **in English**. Where a question asks for a value, write the value only.
   Every number you need is given in the question or in a Check line.
5. `Runtime ▸ Restart and run all` must finish without an error. **Keep the outputs** in the notebook when you submit.
6. Submit the `.ipynb` file to **"Programming Assignment 6 – Sentiment Analysis"** under *Assignments*.

**10 TODO items (25 blanks, 80 points) and 5 short answers (two 2-point items each, 20 points).** Each TODO lists its points per blank.

| Part | Content | Code | Short answers | Points |
|---|---|---|---|---|
| 1 | Lexicon-based : labels, counting opinion words, evaluation, WordNet, the t-test of a word | TODO 1-5 (38) | 1, 2, 3 (12) | 50 |
| 2 | Machine learning-based : bigrams, word weights, Elastic Net, label spreading, comparison | TODO 6-10 (42) | 4, 5 (8) | 50 |
| | **Total** | **80** | **20** | **100** |

## &nbsp;0. Setup (given)
*No TODO in this section.* Run it once and check the printed numbers: `train : 3000   test : 1000`.

In [ ]:
import re
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression, ElasticNet
from sklearn.metrics import accuracy_score, f1_score, mean_absolute_error

import nltk
for resource in ['opinion_lexicon', 'wordnet', 'omw-1.4']:
    nltk.download(resource, quiet=True)
from nltk.corpus import opinion_lexicon, wordnet as wn

pd.set_option('display.width', 140)
POS_WORDS, NEG_WORDS = set(opinion_lexicon.positive()), set(opinion_lexicon.negative())


def tokenize(text):
    return re.findall(r"[a-z][a-z'-]*", text.lower())

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/snhzyn/2026-2-Unstructured-Data-Analysis/main/week12-sentiment-analysis/data/tripadvisor_assignment.csv"

reviews = pd.read_csv(DATA_URL)
train = reviews[reviews['split'] == 'train'].reset_index(drop=True)
test = reviews[reviews['split'] == 'test'].reset_index(drop=True)

print('train :', len(train), '  test :', len(test))
print('test reviews per rating :', test['rating'].value_counts().sort_index().to_dict())

## &nbsp;1. Lexicon-based approach (50 pts : code 38 + short answers 12)

### TODO 1: Labels from star ratings (6 points, 3 + 3)

Lecture 10-2 p. 25: *4-5 stars as positive and 1-2 stars as negative. Neutral is ignored.* Complete the mapping, then keep
only the test reviews that are **not** neutral (Notebook 1 §1-2).

**Check:** `test : {'positive': 732, 'negative': 160, 'neutral': 108}   evaluated : 892`

In [ ]:
# TODO 1 (6 pts): star -> sentiment, and drop the neutral reviews

STAR_TO_SENTIMENT = {1: 'negative', 2: 'negative', 3: ________, 4: 'positive', 5: 'positive'}
train['sentiment'] = train['rating'].map(STAR_TO_SENTIMENT)
test['sentiment'] = test['rating'].map(STAR_TO_SENTIMENT)

test_pn = test[test['sentiment'] ________ 'neutral'].reset_index(drop=True)      # neutral is ignored
y_true = test_pn['sentiment']
print('test :', test['sentiment'].value_counts().to_dict(), '  evaluated :', len(test_pn))

### TODO 2: Count the opinion words (9 points, 3 + 3 + 3)

The procedure of lecture 11 p. 44: count the **unique** positive and negative words of a text, and assign it to the side
with more words. A tie goes to *negative*, as in Notebook 1 §2-2.

**Check:**
```
(4, 0) positive | friendly staff, spacious clean room, great breakfast
(1, 2) negative | room dirty, staff rude, not worth price
(1, 1) negative | location good, room noisy
```

In [ ]:
# TODO 2 (9 pts): unique words, the words they share with a list, and the decision

def count_opinion_words(text, pos_words, neg_words):
    words = ________(tokenize(text))                      # unique words of the text
    return len(words ________ pos_words), len(words & neg_words)


def lexicon_classify(text, pos_words, neg_words):
    n_pos, n_neg = count_opinion_words(text, pos_words, neg_words)
    return 'positive' if n_pos ________ n_neg else 'negative'


for s in ['friendly staff, spacious clean room, great breakfast',
          'room dirty, staff rude, not worth price',
          'location good, room noisy']:
    print(count_opinion_words(s, POS_WORDS, NEG_WORDS), lexicon_classify(s, POS_WORDS, NEG_WORDS), '|', s)

### Short answer 1 (4 points, 2 + 2)

(a) [2 pts] In the Check of TODO 2, the third sentence has one positive and one negative word, and `lexicon_classify`
returns *negative*. Why?

(b) [2 pts] The sentence *"the staff were nice, but the room was hardly the best"* gets the counts (2, 0) and is classified
as *positive*, although the writer is complaining about the room. Why does counting opinion words get this sentence wrong?

**Answer (a):**

**Answer (b):**

### TODO 3: Evaluate the opinion lexicon (8 points, 4 + 4)

Classify every evaluated test review with the opinion lexicon, then compute the accuracy and the F1 score of **each**
class (one number per class, not an average).

**Check:** `accuracy 0.897   F1 negative 0.667   F1 positive 0.939`

In [ ]:
# TODO 3 (8 pts): accuracy and the F1 score per class

pred_lexicon = [lexicon_classify(t, POS_WORDS, NEG_WORDS) for t in test_pn['text']]
acc = ________(y_true, pred_lexicon)
f1_neg, f1_pos = f1_score(y_true, pred_lexicon, labels=['negative', 'positive'], average=________)

print(f'accuracy {acc:.3f}   F1 negative {f1_neg:.3f}   F1 positive {f1_pos:.3f}')

### Short answer 2 (4 points, 2 + 2)

(a) [2 pts] In TODO 1, 732 of the 892 evaluated test reviews are positive. What accuracy does a classifier that always
answers *positive* get on these reviews? Write the value only. A fraction is accepted.

(b) [2 pts] The opinion lexicon reaches an accuracy of 0.897, but the F1 score of the negative class is only 0.667.
Explain why.

**Answer (a):**

**Answer (b):**

### TODO 4: Grow a lexicon from WordNet (6 points, 3 + 3)

Lecture 11 p. 11: start from a small seed set and use WordNet's synsets. A synonym keeps the polarity of the seed, an
antonym gets the opposite polarity (Notebook 1 §2-5).

**Check:**
```
step 1 : positive 12   negative 5   in both 0
step 2 : positive 33   negative 9   in both 0
positive after step 2 : ['amicable', 'booming', 'comfortable', 'comfy', 'congenial', 'easy', 'easygoing', 'favorable', 'favourable', 'flourishing']
```

In [ ]:
# TODO 4 (6 pts): adjective synsets of a word, and the antonyms of a lemma

def expand_with_wordnet(pos, neg):
    new_pos, new_neg = set(pos), set(neg)
    for words, same, opposite in [(pos, new_pos, new_neg), (neg, new_neg, new_pos)]:
        for word in words:
            for synset in wn.________(word, pos=wn.ADJ):       # adjective senses of the word
                for lemma in synset.lemmas():
                    same.add(lemma.name())                      # synonym -> same polarity
                    for antonym in lemma.________():
                        opposite.add(antonym.name())            # antonym -> opposite polarity
    return new_pos, new_neg


pos, neg = {'comfortable', 'friendly'}, {'uncomfortable', 'unfriendly'}
for step in [1, 2]:
    pos, neg = expand_with_wordnet(pos, neg)
    print(f'step {step} : positive {len(pos)}   negative {len(neg)}   in both {len(pos & neg)}')
print('positive after step 2 :', sorted(pos)[:10])

### TODO 5: The t-test of one word (9 points, 3 + 3 + 3)

Lecture 11 p. 18-19: compare the ratings of the training reviews **with** the word and **without** the word
(Welch's t-test, Notebook 1 §3-1). Use the sample variance (divided by $n - 1$).

$$T_w = \frac{E(w) - E(W)}{\sqrt{s_w^2 / n(w) + s_W^2 / n(W)}}$$

**Check:**
```
wonderful  E(w) 4.61  n(w)  405   E(W) 3.87   T   17.43  -> Positive
good       E(w) 3.95  n(w) 1320   E(W) 3.99   T   -0.99  -> Neutral
pool       E(w) 4.00  n(w)  461   E(W) 3.96   T    0.69  -> Neutral
smell      E(w) 2.92  n(w)   66   E(W) 3.99   T   -6.27  -> Negative
```

In [ ]:
# TODO 5 (9 pts): without-word ratings, sample variance, the T statistic

ratings = train['rating'].to_numpy(dtype=float)        # all 3,000 training reviews


def t_test_word(word, alpha=0.05):
    has = train['text'].map(lambda t: word in set(tokenize(t))).to_numpy()
    with_w, without_w = ratings[has], ratings[________]
    a = with_w.var(ddof=________) / len(with_w)                # s_w^2 / n(w)
    b = without_w.var(ddof=1) / len(without_w)          # s_W^2 / n(W)
    T = (with_w.mean() - without_w.mean()) / np.________(a + b)
    v = (a + b) ** 2 / (a ** 2 / (len(with_w) - 1) + b ** 2 / (len(without_w) - 1))
    t_crit = stats.t.ppf(1 - alpha / 2, v)
    diag = 'Positive' if T > t_crit else 'Negative' if T < -t_crit else 'Neutral'
    print(f'{word:10} E(w) {with_w.mean():.2f}  n(w) {len(with_w):4}   E(W) {without_w.mean():.2f}   T {T:7.2f}  -> {diag}')


for word in ['wonderful', 'good', 'pool', 'smell']:
    t_test_word(word)

### Short answer 3 (4 points, 2 + 2)

(a) [2 pts] A word w occurs in n(w) = 50 reviews with E(w) = 3.5 and $s_w^2$ = 1.92, and the other n(W) = 1,000 reviews
have E(W) = 4.0 and $s_W^2$ = 1.6. Compute $T_w$ with the formula of TODO 5, and give its Diag when $t_{(\alpha; v)}$ = 2.0.
Write the two answers only.

(b) [2 pts] *good* is in the positive list of the opinion lexicon, but in the Check of TODO 5 its T is −0.99 and its Diag
is *Neutral*. Why does the t-test not label *good* as Positive in these hotel reviews?

**Answer (a):**

**Answer (b):**

## &nbsp;2. Machine learning-based approach (50 pts : code 42 + short answers 8)

### TODO 6: Logistic regression with unigrams and bigrams (9 points, 3 + 3 + 3)

Lecture 10-2 p. 25: sentiment classification is a text classification problem, and *negations* are a useful feature.
Count unigrams **and** bigrams (Notebook 2 §1-2), count the test reviews with the training vocabulary, and predict.

**Check:** `features 28695   accuracy 0.942   F1 negative 0.824`

In [ ]:
# TODO 6 (9 pts): unigrams + bigrams, the test matrix, the predictions

train_pn = train[train['sentiment'] != 'neutral'].reset_index(drop=True)      # (given)
y_train = train_pn['sentiment']

vec = CountVectorizer(min_df=2, ngram_range=(1, ________))
X_train = vec.fit_transform(train_pn['text'])
X_test = vec.________(test_pn['text'])                   # same vocabulary as training

lr = LogisticRegression(max_iter=2000).fit(X_train, y_train)
pred_lr = lr.________(X_test)
f1_neg_lr = f1_score(y_true, pred_lr, labels=['negative', 'positive'], average=None)[0]
print(f'features {X_train.shape[1]}   accuracy {accuracy_score(y_true, pred_lr):.3f}   F1 negative {f1_neg_lr:.3f}')

### TODO 7: The words with the largest weights (6 points, 3 + 3)

Order the features by their logistic regression weight and print the 8 most negative and the 8 most positive ones,
the most positive first (Notebook 2 §1-1).

**Check:**
```
most negative : ['bad', 'dirty', 'terrible', 'not stay', 'horrible', 'rude', 'worst', 'not']
most positive : ['great', 'wonderful', 'loved', 'fantastic', 'friendly', 'lovely', 'helpful', 'excellent']
```

In [ ]:
# TODO 7 (6 pts): sort the weights, then read the order from both ends

names = vec.get_feature_names_out()
order = np.________(lr.coef_[0])                           # from the most negative to the most positive weight
print('most negative :', list(names[order[:8]]))
print('most positive :', list(names[order[________][:8]]))

### TODO 8: The Elastic Net lexicon (9 points, 3 + 3 + 3)

Lecture 11 p. 22-24: the rating is the response, the words are the explanatory variables, and the coefficient of a word
is its sentiment score (Notebook 2 §2). The penalty is given (`alpha=0.003`, `l1_ratio=0.5`). Fit the model on the
ratings, count the words that kept a non-zero score, and predict the ratings of the test reviews.
A positive coefficient means that the word goes with higher ratings in these reviews, not that its dictionary meaning is
positive (*flat*, British English for an apartment, is one example).

**Check:**
```
lexicon : 756 words   MAE 0.682   COR 0.727
most negative : ['worst', 'terrible', 'unhelpful', 'unfriendly', 'dirty']   most positive : ['fantastic', 'flat', 'wonderful', 'loved', 'great']
```

In [ ]:
# TODO 8 (9 pts): fit on the ratings, count the lexicon, predict the test ratings

vec_bin = CountVectorizer(binary=True, min_df=5)
D_train = vec_bin.fit_transform(train['text'])              # all 3,000 training reviews, 3 stars included
D_test = vec_bin.transform(test['text'])

en = ElasticNet(alpha=0.003, l1_ratio=0.5, max_iter=5000).fit(D_train, train[________])
coef = pd.Series(en.coef_, index=vec_bin.get_feature_names_out())
rating_pred = en.________(D_test)

print(f'lexicon : {(coef ________ 0).sum()} words   MAE {mean_absolute_error(test["rating"], rating_pred):.3f}   '
      f'COR {np.corrcoef(test["rating"], rating_pred)[0, 1]:.3f}')
print('most negative :', list(coef.nsmallest(5).index), '  most positive :', list(coef.nlargest(5).index))

### Short answer 4 (4 points, 2 + 2)

(a) [2 pts] Using the Check of TODO 8, how many of the 4,187 words have a coefficient of exactly 0? Write the value only.

(b) [2 pts] Which part of the Elastic Net penalty sets these coefficients to exactly 0, and why is that useful when the
coefficients are used as a sentiment lexicon?

**Answer (a):**

**Answer (b):**

### TODO 9: Label spreading on the lecture's word graph (12 points, 4 + 4 + 4)

Lecture 11 p. 33-34 (Notebook 2 §3-1). Every step, each word takes the average score of its neighbors, and the seeds keep
pulling toward their label:

$$F \leftarrow \alpha \, D^{-1} W \, F + (1 - \alpha) \, Y$$

The graph is the one of the lecture, but the seeds are **different** from the notebook: two positive and two negative words.

**Check:**
```
after 1 step(s)  positive: ['awesome', 'entertaining', 'good', 'heartfelt', 'song']
                  negative: ['annoying', 'dirty', 'disappointing', 'weak']
after 2 step(s)  positive: ['awesome', 'burst', 'entertaining', 'good', 'handsome', 'heartfelt', 'sad', 'song', 'watch']
                  negative: ['annoying', 'camera', 'dirty', 'disappointing', 'forced', 'friend', 'insufficient', 'mediocre', 'superficial', 'weak']
watch after 30 steps : 0.031
```

In [ ]:
# TODO 9 (12 pts): degrees, one step of spreading

EDGES = [('song', 'burst'), ('song', 'fun'), ('burst', 'good'), ('good', 'awesome'), ('good', 'fun'),
         ('good', 'entertaining'), ('fun', 'entertaining'), ('awesome', 'sad'), ('awesome', 'excellent'),
         ('excellent', 'entertaining'), ('excellent', 'heartfelt'), ('sad', 'director'), ('sad', 'heartfelt'),
         ('heartfelt', 'handsome'), ('entertaining', 'handsome'), ('handsome', 'protagonist'),
         ('entertaining', 'watch'), ('watch', 'forced'), ('forced', 'mediocre'), ('mediocre', 'annoying'),
         ('superficial', 'annoying'), ('forced', 'annoying'), ('annoying', 'lacking'), ('lacking', 'disappointing'),
         ('annoying', 'boring'), ('forced', 'dirty'), ('dirty', 'boring'), ('dirty', 'friend'),
         ('boring', 'disappointing'), ('disappointing', 'insufficient'), ('boring', 'weak'),
         ('insufficient', 'weak'), ('weak', 'camera')]                       # (given)
SEEDS = {'excellent': 1, 'fun': 1, 'boring': -1, 'lacking': -1}

toy_words = sorted({w for e in EDGES for w in e})
idx = {w: i for i, w in enumerate(toy_words)}
W = np.zeros((len(toy_words), len(toy_words)))
for a, b in EDGES:
    W[idx[a], idx[b]] = W[idx[b], idx[a]] = 1
y = np.array([SEEDS.get(w, 0) for w in toy_words], dtype=float)


def spread(W, y, alpha=0.8, n_steps=30):
    deg = W.________(axis=1)                                   # number of neighbors of each word
    P = W / deg[:, None]                                    # D^-1 W : average over the neighbors
    F = y.copy()
    for _ in range(n_steps):
        F = alpha * (P ________ F) + (1 - alpha) * ________
    return F


for steps in [1, 2]:
    F = pd.Series(spread(W, y, n_steps=steps), index=toy_words)
    print(f'after {steps} step(s)  positive: {sorted(F.index[(F > 0.01) & (y == 0)])}')
    print(f'{"":18}negative: {sorted(F.index[(F < -0.01) & (y == 0)])}')
print('watch after 30 steps :', round(spread(W, y)[idx['watch']], 3))

### Short answer 5 (4 points, 2 + 2)

(a) [2 pts] *song* has two neighbors, *burst* and *fun*, and every score starts at its seed label (+1, −1 or 0). What is the
score of *song* after one step of `spread` with $\alpha$ = 0.8? Write the value only.

(b) [2 pts] *camera* is connected only to *weak*, which is connected to *boring*. Why does *camera* get a negative score
only after two steps?

**Answer (a):**

**Answer (b):**

### TODO 10: The Elastic Net lexicon as a classifier, and the comparison (6 points, 3 + 3)

A predicted rating of **3 or more** is read as *positive* (Notebook 2 §2). Then put the three methods of this assignment
in one table, sorted by accuracy from the lowest to the highest.

**Check:**
```
                            accuracy  F1 negative
Opinion lexicon (count)        0.897        0.667
Elastic Net lexicon            0.918        0.738
Logistic regression (1, 2)     0.942        0.824
```

In [ ]:
# TODO 10 (6 pts): threshold the predicted ratings, sort the table

is_pn = (test['sentiment'] != 'neutral').to_numpy()
pred_en = np.where(rating_pred[is_pn] ________ 3, 'positive', 'negative')

comparison = pd.DataFrame({
    'accuracy': [accuracy_score(y_true, p) for p in [pred_lexicon, pred_en, pred_lr]],
    'F1 negative': [f1_score(y_true, p, labels=['negative', 'positive'], average=None)[0]
                    for p in [pred_lexicon, pred_en, pred_lr]],
}, index=['Opinion lexicon (count)', 'Elastic Net lexicon', 'Logistic regression (1, 2)'])
print(comparison.________('accuracy').round(3))

---
### ✅ Before you submit

- [ ] `Runtime ▸ Restart and run all` finishes with no error
- [ ] Every `________` is filled in and every output cell is visible
- [ ] Every **Check** line of the assignment matches your output
- [ ] All five short answers are written **in English** in their answer cells
- [ ] The file is named `Assignment_6_YourName_StudentID.ipynb`